# Simple Event Log Analysis
#### Task:
Report basic event log statistics in a table, inclunding:
- number of cases
- number of events
- number of process variants
- number of process variants to cover 80% of cases 
- number of distinct case and event attribute labels 
- mean and standard deviation of case length
- mean and standard deviation of case duration
-  number of categorical event attributes
 

#### Import relevant libraries

In [1]:
import pm4py # process mining library
import pandas as pd # data manipulation library
import os # operating system library for file handling
import matplotlib.pyplot as plt # plotting library for visualizations
import numpy as np # numerical computing library for data analysis

**load log**

In [5]:
raw_log = pm4py.read_xes("../data/BPIChallenge2017.xes.gz")

parsing log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]

#### define analysis function:
- number of cases
- number of events
- number of process variants
- number of process variants to cover 80% of cases 
- mean and standard deviation of case length
- mean and standard deviation of case duration

In [25]:
def get_event_log_info(log):
    try:
        # Calculate basic statistics about the event log
        num_cases = log['case:concept:name'].nunique()
        num_events = log['EventID'].nunique()

        # Calculate the number of unique process variants
        num_process_variants = len(pm4py.get_variants(log))

        # Calculate coverage of the most common process variant
        variants = pm4py.get_variants(log)
        variant_counts = pd.Series(variants).sort_values(ascending=False)
        variant_coverage = variant_counts.cumsum() / variant_counts.sum()
        num_variants_covered_80_percent = (variant_coverage <= 0.8).sum() + 1  # +1 to include the variant that reaches 80%

        # Get length of each case == number of events per case
        len_cases = log.groupby('case:concept:name').size()
        mean_case_length = len_cases.mean()
        std_case_length = len_cases.std()

        duration_cases = pm4py.stats.get_all_case_durations(log)
        mean_case_duration = np.mean(duration_cases)
        std_case_duration = np.std(duration_cases)

        def format_duration(seconds):
            if seconds < 60:
                return f"{seconds:.2f} seconds"
            elif seconds < 3600:
                return f"{seconds / 60:.2f} minutes"
            elif seconds < 86400:
                return f"{seconds / 3600:.2f} hours"
            else:
                return f"{seconds / 86400:.2f} days"

        stats = pd.DataFrame({
            'Metric': [
                'Number of cases',
                'Number of events',
                'Number of process variants',
                'Number of variants covering 80% of cases',
                'Mean case length',
                'Std case length',
                'Mean case duration',
                'Std case duration',
            ],
            'Value': [
                num_cases,
                num_events,
                num_process_variants,
                num_variants_covered_80_percent,
                mean_case_length,
                std_case_length,
                format_duration(mean_case_duration),
                format_duration(std_case_duration)
            ]
        }).set_index('Metric')

        return stats

    except Exception as e:
        print(f"Error processing event log: {e}")
        return None

#### define analysis function:
*(automatic interpretation)*
- distinct case and attribute labels
-  number of categorical event attributes

In [29]:
def get_attributes(log):
    try:
        case_attr_labels = [col for col in log.columns if col.startswith('case:')]
        event_attr_labels = [col for col in log.columns if not col.startswith('case:')]

        num_case_attr_labels = len(case_attr_labels)
        num_event_attr_labels = len(event_attr_labels)

        event_cols = [col for col in log.columns if not col.startswith('case:')]
        num_categorical_event_attrs = sum(log[col].dtype in ('object', 'category') for col in event_cols)
        stats = pd.DataFrame({
            'Metric': [
                'Number of case attribute labels',
                'Number of event attribute labels',
                'Number of categorical event attributes'
            ],
            'Value': [
                num_case_attr_labels,
                num_event_attr_labels,
                num_categorical_event_attrs
            ]
        }).set_index('Metric')
        return stats
    except Exception as e:
        print(f"Error processing event log: {e}")
        return None

- analyze raw log: 

In [30]:
get_event_log_info(raw_log)

,Value
Metric,
Number of cases,31509
Number of events,1202267
Number of process variants,15930
Number of variants covering 80% of cases,9629
Mean case length,38.156305
Std case length,16.715308
Mean case duration,21.90 days
Std case duration,13.17 days


In [31]:
get_attributes(raw_log)

,Value
Metric,
Number of case attribute labels,4
Number of event attribute labels,15
Number of categorical event attributes,2
